In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from flask import Flask, request, jsonify

# 1. Generate Historical Training Data
def generate_mock_training_data(num_samples=1000):
    np.random.seed(42)
    soil_moisture = np.random.uniform(10, 90, num_samples)       # Percentage (%)
    temperature = np.random.uniform(15, 35, num_samples)         # Celsius (°C)
    humidity = np.random.uniform(30, 90, num_samples)            # Percentage (%)
    time_since_watered = np.random.uniform(1, 168, num_samples)  # Hours

    needs_water = []
    for sm, temp, hum, tsw in zip(soil_moisture, temperature, humidity, time_since_watered):
        if sm < 30 or (sm < 50 and tsw > 48) or (temp > 30 and sm < 45):
            needs_water.append(1) # Needs Water
        else:
            needs_water.append(0) # Does Not Need Water

    df = pd.DataFrame({
        'soil_moisture': soil_moisture,
        'temperature': temperature,
        'humidity': humidity,
        'time_since_watered': time_since_watered,
        'needs_water': needs_water
    })
    return df

# 2. Train the AI Model
def train_model():
    df = generate_mock_training_data()

    X = df[['soil_moisture', 'temperature', 'humidity', 'time_since_watered']]
    y = df['needs_water']

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)

    predictions = model.predict(X_test)
    print(f"AI Model Trained Successfully! Accuracy: {accuracy_score(y_test, predictions) * 100:.2f}%")

    return model

# 3. Real-time Prediction Function
def predict_watering_action(model, current_soil_moisture, current_temp, current_humidity, hours_since_watered):
    input_data = np.array([[current_soil_moisture, current_temp, current_humidity, hours_since_watered]])
    prediction = model.predict(input_data)
    probability = model.predict_proba(input_data)

    action = "WATER_PLANT" if prediction[0] == 1 else "DO_NOTHING"
    confidence = np.max(probability) * 100

    return {"action": action, "confidence": f"{confidence:.2f}%"}

# 4. Initialize Flask App & AI Model
app = Flask(__name__)
print("Initializing AI Smart Plant Care Backend...")
model = train_model()

@app.route('/api/sensor-data', methods=['POST'])
def handle_sensor_data():
    try:
        data = request.get_json()

        soil_moisture = float(data.get('soil_moisture'))
        temperature = float(data.get('temperature'))
        humidity = float(data.get('humidity'))
        hours_since_watered = float(data.get('hours_since_watered', 24))

        # Get decision from AI model
        result = predict_watering_action(model, soil_moisture, temperature, humidity, hours_since_watered)

        response = {
            "status": "success",
            "ai_decision": result['action'],
            "confidence": result['confidence']
        }

        return jsonify(response), 200

    except Exception as e:
        return jsonify({"status": "error", "message": str(e)}), 400

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5000, debug=True)

Initializing AI Smart Plant Care Backend...
AI Model Trained Successfully! Accuracy: 100.00%
 * Serving Flask app '__main__'
 * Debug mode: on


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug: * Restarting with watchdog (inotify)
